# Task 5 - Auto-Tagging Support Tickets Using an LLM
**DevelopersHub Corporation - AI/ML Engineering Internship**

## Problem Statement & Objective
Customer support teams receive thousands of free-text tickets. Reading each one to
route it to the right team (Billing, Technical, etc.) is slow. **Objective:** use a
Large Language Model (LLM) to automatically read a ticket and assign the correct
**category tag** - and return the **top 3 most likely tags** per ticket.

## Approach (what makes this different from a normal ML model)
In a normal ML task we *train* a model on labelled data (we change its weights).
Here we do **NOT train anything**. We use *pre-trained* LLMs and just instruct them.
We will use two techniques and compare them:

1. **Zero-shot classification** - we give the model the ticket + a list of possible
   categories. The model has never seen our categories before, yet it can pick the
   right one using its general language understanding.
2. **Few-shot prompting** - we put a few *example* (ticket -> category) pairs inside
   the prompt. The model "learns" the pattern from these examples *in the prompt
   itself* (called **in-context learning**) - still no weight training.

> Note: the task mentions "fine-tuning" as one option. True fine-tuning of an LLM needs
> heavy compute and lots of labelled data. The modern, practical alternative - and what
> we compare here - is **prompt engineering: zero-shot vs few-shot**.


## 1. Setup
We update `transformers` and import what we need. `torch` is already in Colab.
**Tip:** turn on a GPU for speed -> Runtime > Change runtime type > T4 GPU.

In [ ]:
!pip install -q -U transformers

import torch
import pandas as pd
import matplotlib.pyplot as plt
from transformers import pipeline
from sklearn.metrics import accuracy_score, classification_report

# Use GPU if available (much faster), else CPU
DEVICE = 0 if torch.cuda.is_available() else -1
print("Running on:", "GPU" if DEVICE == 0 else "CPU")

## 2. The Support-Ticket Dataset
We use a small, realistic set of free-text tickets, each with a known **true category**
(the "ground truth") so we can *measure* how well the LLM tags them.

> You can later swap this for a larger Kaggle "Customer Support Tickets" dataset -
> just load it into a DataFrame with `text` and `category` columns and the rest works.

In [ ]:
tickets = [
    ("I was charged twice for my monthly subscription, please fix this.", "Billing"),
    ("My invoice shows an amount higher than what I agreed to.", "Billing"),
    ("Why was my credit card billed when I cancelled last week?", "Billing"),
    ("I need a copy of my last three payment receipts.", "Billing"),
    ("There is an unexpected service fee on my latest bill.", "Billing"),
    ("Can you explain the extra charges added to my account this month?", "Billing"),

    ("The mobile app keeps crashing whenever I open the dashboard.", "Technical Issue"),
    ("I get a 500 error every time I try to upload a file.", "Technical Issue"),
    ("The website is extremely slow and pages will not load.", "Technical Issue"),
    ("Video playback freezes after a few seconds on my laptop.", "Technical Issue"),
    ("The export button does nothing when I click it.", "Technical Issue"),
    ("I keep getting disconnected from the server randomly.", "Technical Issue"),

    ("I forgot my password and the reset link never arrives.", "Account Access"),
    ("My account is locked after too many login attempts.", "Account Access"),
    ("Two-factor authentication is not sending me a code.", "Account Access"),
    ("I cannot log in even though my password is correct.", "Account Access"),
    ("How do I change the email address on my account?", "Account Access"),
    ("My account was suspended and I do not know why.", "Account Access"),

    ("My order has not arrived even though it shows delivered.", "Shipping"),
    ("Can I change the delivery address for my recent order?", "Shipping"),
    ("The tracking number you sent me is not working.", "Shipping"),
    ("My package is stuck in transit for over a week.", "Shipping"),
    ("I received the wrong item in my shipment.", "Shipping"),
    ("When will my order be dispatched?", "Shipping"),

    ("I want a refund for the product I returned last month.", "Refund"),
    ("How long does it take to process a refund?", "Refund"),
    ("The item was damaged, I would like my money back.", "Refund"),
    ("I returned the package but have not received my refund yet.", "Refund"),
    ("Can I get a refund if I cancel within 24 hours?", "Refund"),
    ("I was promised a refund but it never showed up.", "Refund"),

    ("Does your premium plan include team collaboration features?", "Product Inquiry"),
    ("How do I set up automatic backups in the app?", "Product Inquiry"),
    ("Is there a way to integrate your tool with Slack?", "Product Inquiry"),
    ("What is the difference between the basic and pro plans?", "Product Inquiry"),
    ("Can I use this software on multiple devices?", "Product Inquiry"),
    ("How do I enable dark mode in the settings?", "Product Inquiry"),
]

df = pd.DataFrame(tickets, columns=["text", "category"])
CATEGORIES = sorted(df["category"].unique().tolist())

print("Total tickets:", len(df))
print("Categories:", CATEGORIES)
df["category"].value_counts()

## 3. Approach 1 - Zero-Shot Classification
We load `facebook/bart-large-mnli`. This model was trained to judge whether one
sentence *implies* another. We cleverly reuse it for classification: it checks
"does this ticket imply the topic is *Billing*? *Shipping*? ..." and scores each
category. The category with the highest score wins - **no training on our data needed.**

In [ ]:
zero_shot = pipeline("zero-shot-classification",
                     model="facebook/bart-large-mnli",
                     device=DEVICE)
print("Zero-shot model loaded.")

### Try it on one ticket and look at the ranked scores
The pipeline returns every category with a probability. We take the **top 3** as our
suggested tags.

In [ ]:
sample_text = "My payment failed twice but money was still deducted."
result = zero_shot(sample_text, candidate_labels=CATEGORIES)

print("Ticket:", sample_text, "\n")
for label, score in zip(result["labels"][:3], result["scores"][:3]):
    print(f"  {label:18s} -> {score:.3f}")

## 4. Output the Top-3 Tags for Every Ticket
We loop over all tickets and store: the **top prediction** (for accuracy) and the
**top-3 tags** (the deliverable).

In [ ]:
def top3_tags(text):
    r = zero_shot(text, candidate_labels=CATEGORIES)
    top3 = [(lab, round(sc, 3)) for lab, sc in zip(r["labels"][:3], r["scores"][:3])]
    return r["labels"][0], top3   # (best_label, list_of_top3)

zs_preds, zs_top3 = [], []
for t in df["text"]:
    best, t3 = top3_tags(t)
    zs_preds.append(best)
    zs_top3.append(t3)

df["zero_shot_pred"] = zs_preds
df["top_3_tags"] = zs_top3

# Show a few tickets with their top-3 tags
df[["text", "category", "zero_shot_pred", "top_3_tags"]].head(8)

### Evaluate the zero-shot approach
We compare the model's top prediction against the true category.

In [ ]:
zs_acc = accuracy_score(df["category"], df["zero_shot_pred"])
print(f"Zero-shot accuracy: {zs_acc:.1%}\n")
print(classification_report(df["category"], df["zero_shot_pred"], zero_division=0))

## 5. Approach 2 - Few-Shot Prompting with a Generative LLM
Now we use `google/flan-t5-base`, an instruction-following model. We compare two prompts:

- **Zero-shot prompt:** just the instruction + categories.
- **Few-shot prompt:** the same, *plus* 3 worked examples.

The model reads the examples in the prompt and imitates the pattern. This is
**in-context learning** - the heart of modern prompt engineering.

> We load the model **directly** (tokenizer + model) instead of via `pipeline(...)`.
> This is more robust: it does not depend on pipeline task-name aliases, which can
> change between `transformers` versions.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

gen_name = "google/flan-t5-base"
gen_tokenizer = AutoTokenizer.from_pretrained(gen_name)
gen_model = AutoModelForSeq2SeqLM.from_pretrained(gen_name)
if DEVICE == 0:
    gen_model = gen_model.to("cuda")

def generate_text(prompt, max_new_tokens=10):
    inputs = gen_tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)
    if DEVICE == 0:
        inputs = {k: v.to("cuda") for k, v in inputs.items()}
    out_ids = gen_model.generate(**inputs, max_new_tokens=max_new_tokens)
    return gen_tokenizer.decode(out_ids[0], skip_special_tokens=True)

print("Generative model loaded.")

### Build the two prompts
The few-shot examples are written separately (they are NOT from our test tickets),
so we are not "leaking" test answers into the prompt.

In [ ]:
cats_str = ", ".join(CATEGORIES)

# 3 example pairs used ONLY for the few-shot prompt (not in the dataset)
few_shot_examples = [
    ("I keep getting charged after cancelling my plan.", "Billing"),
    ("The app freezes every time I open my profile.", "Technical Issue"),
    ("I cannot reset my password, no email comes through.", "Account Access"),
]

def zero_shot_prompt(text):
    return (f"Classify this customer support ticket into exactly one category "
            f"from: {cats_str}.\nTicket: {text}\nCategory:")

def few_shot_prompt(text):
    p = f"Classify the customer support ticket into exactly one category from: {cats_str}.\n\n"
    for ex_text, ex_cat in few_shot_examples:
        p += f"Ticket: {ex_text}\nCategory: {ex_cat}\n\n"
    p += f"Ticket: {text}\nCategory:"
    return p

print(few_shot_prompt("My package never arrived."))

### Helper: turn the model's free text into one of our categories
The model might write "Billing." or "billing dept" - we map any output back to the
closest official category.

In [ ]:
def match_category(output_text):
    out = output_text.strip().lower()
    # 1) direct match: a category name appears in the output
    for c in CATEGORIES:
        if c.lower() in out:
            return c
    # 2) fallback: category sharing the most words with the output
    best, best_score = CATEGORIES[0], -1
    for c in CATEGORIES:
        overlap = len(set(c.lower().split()) & set(out.split()))
        if overlap > best_score:
            best_score, best = overlap, c
    return best

def predict(prompt_fn):
    preds = []
    for t in df["text"]:
        out = generate_text(prompt_fn(t))
        preds.append(match_category(out))
    return preds

### Run both prompts and measure accuracy

In [ ]:
zs_prompt_preds = predict(zero_shot_prompt)
fs_prompt_preds = predict(few_shot_prompt)

zs_prompt_acc = accuracy_score(df["category"], zs_prompt_preds)
fs_prompt_acc = accuracy_score(df["category"], fs_prompt_preds)

print(f"Flan-T5 zero-shot prompt accuracy: {zs_prompt_acc:.1%}")
print(f"Flan-T5 few-shot  prompt accuracy: {fs_prompt_acc:.1%}")

## 6. Compare the Approaches

In [ ]:
labels = ["BART\nzero-shot", "Flan-T5\nzero-shot prompt", "Flan-T5\nfew-shot prompt"]
scores = [zs_acc, zs_prompt_acc, fs_prompt_acc]

plt.figure(figsize=(7,4))
bars = plt.bar(labels, scores, color=["#4C72B0", "#DD8452", "#55A868"])
plt.ylabel("Accuracy"); plt.ylim(0, 1); plt.title("Tagging Accuracy by Approach")
for b, s in zip(bars, scores):
    plt.text(b.get_x()+b.get_width()/2, s+0.02, f"{s:.0%}", ha="center")
plt.tight_layout(); plt.show()

## 7. Final Summary & Insights
- Tagged free-text support tickets using **LLMs with no training on our data**.
- **Zero-shot classification** (`bart-large-mnli`) produced the **top-3 tags** per
  ticket with confidence scores - useful when a ticket could fit more than one team.
- Compared **zero-shot vs few-shot prompting** with `flan-t5-base`. Few-shot adds a
  few examples to the prompt (**in-context learning**) to steer the model.

*(Fill in your own accuracy numbers from the run above.)* Typically few-shot prompting
matches or beats plain zero-shot, because the examples make the task and the exact
label names crystal clear to the model.

### Key concepts demonstrated
- Prompt engineering (how wording + examples change results)
- Zero-shot and few-shot (in-context) learning
- LLM-based multi-class classification with **top-3 ranking**

### Possible next steps
- Add more few-shot examples or rephrase categories for higher accuracy.
- Try a larger model (`flan-t5-large`) or an instruct API for tougher tickets.
